# 11 - Parallel dashboard metrics refresh

In [ ]:
import os
CATALOG=os.getenv('AIDP_CATALOG','aidp_poc'); parallelism=spark.sparkContext.defaultParallelism
spark.conf.set('spark.sql.adaptive.enabled','true'); spark.conf.set('spark.sql.shuffle.partitions',str(max(200,parallelism*4)))
print('Executor parallelism=',parallelism,'shuffle partitions=',spark.conf.get('spark.sql.shuffle.partitions'))
metrics=spark.sql(f'''WITH bronze AS (SELECT COUNT(*) bronze_events,COUNT(DISTINCT stream_key) bronze_active_meters,MAX(ingested_at) latest_ingested FROM {CATALOG}.bronze.meter_reading),silver AS (SELECT COUNT(*) silver_readings,COUNT(DISTINCT meter_id) active_meters,COUNT(DISTINCT interval_start_utc) intervals,MAX(interval_start_utc) latest_interval FROM {CATALOG}.silver.interval_reading),gold AS (SELECT ROUND(SUM(consumption_kwh),2) latest_kwh,ROUND(MAX(demand_kw),2) peak_kw,SUM(tamper_reading_count) tamper_alerts,SUM(outage_reading_count) outage_alerts FROM {CATALOG}.gold.meter_interval_usage WHERE interval_start_utc=(SELECT MAX(interval_start_utc) FROM {CATALOG}.gold.meter_interval_usage)),ml AS (SELECT COUNT(*) prediction_count,MAX(model_version) model_version,ROUND(AVG(prediction_kwh),4) avg_prediction_kwh,MAX(scored_at) latest_scored FROM {CATALOG}.ml.meter_predictions) SELECT current_timestamp() refreshed_at,bronze.*,silver.*,gold.*,ml.* FROM bronze CROSS JOIN silver CROSS JOIN gold CROSS JOIN ml''')
metrics.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{CATALOG}.gold.dashboard_metrics')
bronze_categories=spark.sql(f'''SELECT COALESCE(get_json_object(raw_value,'$.service_point_type'),'UNKNOWN') meter_type,COALESCE(get_json_object(raw_value,'$.customer_segment'),'UNKNOWN') customer_segment,COUNT(*) events,COUNT(DISTINCT stream_key) active_meters FROM {CATALOG}.bronze.meter_reading GROUP BY 1,2 ORDER BY events DESC''')
bronze_categories.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{CATALOG}.gold.dashboard_bronze_categories')
segments=spark.sql(f'''SELECT current_timestamp() refreshed_at,customer_segment,COUNT(*) meters,ROUND(SUM(consumption_kwh),2) kwh,ROUND(MAX(demand_kw),2) peak_kw FROM {CATALOG}.gold.meter_interval_usage WHERE interval_start_utc=(SELECT MAX(interval_start_utc) FROM {CATALOG}.gold.meter_interval_usage) GROUP BY customer_segment''')
segments.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{CATALOG}.gold.dashboard_segments')
predictions=spark.sql(f'''SELECT model_version,meter_id,interval_start_utc,ROUND(prediction_kwh,4) prediction_kwh,scored_at FROM {CATALOG}.ml.meter_predictions ORDER BY scored_at DESC,meter_id LIMIT 100''')
predictions.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{CATALOG}.gold.dashboard_predictions')
print('Dashboard summary tables refreshed.')